# ENSO × rendimento da soja: série histórica (1974–2025)

**Por que este notebook:** as análises de `eda_enso_defasagens.ipynb` usam a chuva das estações automáticas do INMET, que só cobre bem as safras a partir de 2005 (cerca de 20 safras). A relação **ENSO → rendimento** não depende das estações, então pode usar a série inteira da **PAM/IBGE**: **cerca de 50 safras**, com muito mais eventos de El Niño e La Niña.

**Pergunta:** o ENSO se relaciona com o rendimento da soja em cada região? Com qual antecedência? Os padrões encontrados na chuva (D-03 a D-08) aparecem também no rendimento?

**Estrutura:**
1. dados (PAM por município, 1974–2025) e validação do mapeamento para microrregião;
2. anomalia de rendimento sem a tendência tecnológica, por microrregião e por bioma;
3. perfil de janelas do ONI × rendimento, com teste por séries substitutas;
4. comparação por fase do ENSO;
5. estabilidade entre períodos (1975–1999 × 2000–2025);
6. conclusões.

Decisões em `docs/decisoes.md` (D-09 e D-10). Esta é uma **análise de dados**, não um modelo de previsão.

In [ ]:
import time
import warnings
from pathlib import Path

import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import rankdata
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)

PROCESSED = Path("../../data/processed")
RAW = Path("../../data/raw")

P = dict(
    ano_ini=1974, ano_fim=2025,
    area_min_ha=100,             # mesmo filtro da base de modelagem (D-03 do pipeline)
    rendimento_max=6000,         # kg/ha, mesmo filtro da base de modelagem
    min_anos_micro=15,           # anos mínimos para estimar a tendência de uma microrregião
    janela_tendencia_anos=25,    # largura da janela do LOWESS principal (bem mais lenta que o ciclo do ENSO, 2–7 anos)
    seed=42, n_fase=2000, desloc_min_anos=3,
    n_perm=10000,
)
DIVISAO = "Bioma (Pantanal → Cerrado)"   # D-01

## 1. Dados
**Fonte:** PAM/IBGE, Tabela 1612 (soja em grão), **por município**, de 1974 a 2025, pela API do SIDRA. A série oficial por microrregião só existe a partir de 1990, então os municípios são agregados por microrregião com o mesmo método do pipeline: centroide do município na malha de 2020 (geobr).

Os arquivos são baixados uma vez e guardados em `data/raw/` (cerca de 9 minutos na primeira execução).

In [ ]:
ARQ_MUN = RAW / "pam_soja_municipio_1974_2025.csv"
ARQ_OFI = RAW / "pam_soja_microrregiao_oficial_1990_2025.csv"


def baixar_pam(nivel: str, anos) -> pd.DataFrame:
    import sidrapy
    dfs = []
    for a in anos:
        for tentativa in range(3):
            try:
                d = sidrapy.get_table(table_code="1612", territorial_level=nivel, ibge_territorial_code="all",
                                      variable="214,216", classifications={"81": "2713"}, period=str(a)).iloc[1:]
                dfs.append(d[["D1C", "D1N", "D2C", "D3C", "V"]])
                break
            except Exception as e:
                print(f"  tentativa {tentativa + 1} falhou para {a}: {e}")
                time.sleep(3)
        time.sleep(0.5)
    t = pd.concat(dfs)
    t["V"] = pd.to_numeric(t["V"], errors="coerce")
    t = t.pivot_table(index=["D1C", "D1N", "D2C"], columns="D3C", values="V", aggfunc="first").reset_index()
    return t.rename(columns={"D1C": "codigo", "D1N": "nome", "D2C": "ano", "214": "producao_t", "216": "area_colhida_ha"})


RAW.mkdir(parents=True, exist_ok=True)
if not ARQ_MUN.exists():
    baixar_pam("6", range(P["ano_ini"], P["ano_fim"] + 1)).to_csv(ARQ_MUN, index=False)
if not ARQ_OFI.exists():
    baixar_pam("9", range(1990, P["ano_fim"] + 1)).to_csv(ARQ_OFI, index=False)

mun = pd.read_csv(ARQ_MUN, dtype={"codigo": str})
ofi = pd.read_csv(ARQ_OFI, dtype={"codigo": str})
print(f"PAM por município: {mun['ano'].min()}–{mun['ano'].max()} | {mun.dropna(subset=['producao_t'])['codigo'].nunique()} municípios com soja em algum ano")

In [ ]:
# Município -> microrregião (centroide na malha de 2020, como no pipeline) e microrregião -> bioma (D-01)
municipios = geobr.read_municipality(year=2020)
micro = geobr.read_micro_region(year=2020)
micro["CD_MICRO"] = micro["code_micro"].astype(int).astype(str)
cent = municipios.to_crs(5880)
cent["geometry"] = cent.centroid
cent = cent.to_crs(micro.crs)
j = gpd.sjoin(cent[["code_muni", "geometry"]], micro[["CD_MICRO", "geometry"]], predicate="within")
muni_micro = pd.DataFrame({"codigo": j["code_muni"].astype(int).astype(str), "CD_MICRO": j["CD_MICRO"]})

biomas = geobr.read_biomes(year=2019)
biomas = biomas[~biomas["name_biome"].str.contains("Costeiro", case=False, na=False)]
m_ = micro[["CD_MICRO", "geometry"]].to_crs(5880)
b_ = biomas[["name_biome", "geometry"]].to_crs(5880)
m_["geometry"], b_["geometry"] = m_.geometry.make_valid(), b_.geometry.make_valid()
inter = gpd.overlay(m_, b_, how="intersection", keep_geom_type=True)
inter["area"] = inter.area
bioma_micro = (inter.sort_values("area").drop_duplicates("CD_MICRO", keep="last").set_index("CD_MICRO")["name_biome"]
                    .replace({"Pantanal": "Cerrado"}))       # D-01: Pantanal agrupado ao Cerrado

x = mun.merge(muni_micro, on="codigo", how="left")
sem = x[x["CD_MICRO"].isna() & x["producao_t"].notna()]
print(f"Municípios-ano com soja sem microrregião: {len(sem)} ({sem['producao_t'].sum() / x['producao_t'].sum():.3%} da produção)")
micro_ano = x.dropna(subset=["CD_MICRO"]).groupby(["CD_MICRO", "ano"])[["producao_t", "area_colhida_ha"]].sum(min_count=1).reset_index()
micro_ano["bioma"] = micro_ano["CD_MICRO"].map(bioma_micro)

### Validação do mapeamento (1990–2025)
A agregação por município é comparada com a **série oficial do IBGE por microrregião**, nos anos em que as duas existem. A métrica é a fração da produção nacional que cai na mesma microrregião nas duas versões.

**Limitação que esta validação não cobre:** antes de 1990, muitos municípios de fronteira (MT, MATOPIBA) eram maiores e foram desmembrados depois. A produção de um município antigo é atribuída à microrregião da sua sede **atual**, e parte da área original pode estar em outra microrregião. Por isso a análise principal é **por bioma**, onde esse erro entre microrregiões vizinhas praticamente se cancela.

In [ ]:
o = ofi.rename(columns={"codigo": "CD_MICRO"})
v = o.merge(micro_ano, on=["CD_MICRO", "ano"], suffixes=("_ofi", "_agg"))
v = v[(v["producao_t_ofi"].fillna(0) + v["producao_t_agg"].fillna(0)) > 0].fillna({"producao_t_ofi": 0, "producao_t_agg": 0})
v["min"] = np.minimum(v["producao_t_ofi"], v["producao_t_agg"])
conc = v.groupby("ano").apply(lambda g: 100 * g["min"].sum() / g["producao_t_ofi"].sum(), include_groups=False)
print(f"Concordância da produção por microrregião (1990–2025): média {conc.mean():.2f}% | mínima {conc.min():.2f}% ({conc.idxmin()})")

# Participação de cada bioma na produção ao longo do tempo (cobertura dos primeiros anos)
part = micro_ano.groupby(["ano", "bioma"])["producao_t"].sum().unstack().fillna(0)
part = 100 * part.div(part.sum(axis=1), axis=0)
part.loc[[1975, 1980, 1985, 1990, 2000, 2010, 2020, 2025]].round(1)

## 2. Anomalia de rendimento sem a tendência tecnológica
**Por microrregião:**
- rendimento = produção × 1000 ÷ área colhida (kg/ha), com área ≥ 100 ha e rendimento ≤ 6.000 kg/ha (filtros da base de modelagem);
- **tendência tecnológica** por **LOWESS robusto** (regressão local suavizada com 3 iterações que reduzem o peso de anos extremos), com **janela de 25 anos**, ajustada **por microrregião** (é a abordagem por microrregião defendida na revisão do `desvio_clima`). Exige pelo menos 15 anos de dados;
- **anomalia relativa** = rendimento ÷ tendência − 1, em %.

**Por bioma:** média das anomalias das microrregiões, **ponderada pela área colhida**.

**Por que 25 anos e robusto (decisão revista depois da primeira execução):** a primeira versão usava janela de 15 anos. No gráfico de exemplo, a tendência do Pampa **dobrava para baixo no fim da série**, puxada pelas secas seguidas de 2020 a 2023 (anos de La Niña). Uma tendência flexível demais **absorve parte do próprio efeito climático**, justamente nos eventos de vários anos. O princípio adotado é que a tendência tecnológica tem de ser **bem mais lenta que o ciclo do ENSO** (2 a 7 anos) e pouco sensível a anos extremos.

**Sensibilidade ao método de tendência** (seção 2b): LOWESS com janela de 15 anos (a versão original), mediana móvel centrada de 9 anos e polinômio de grau 2 por microrregião.

Aqui, a tendência usa a série inteira, porque o objetivo é **descrever** a relação; não há modelo sendo avaliado. Para modelagem, ela deve ser ajustada só com os anos de treino.

In [ ]:
r = micro_ano.dropna(subset=["producao_t", "area_colhida_ha"]).copy()
r = r[(r["area_colhida_ha"] >= P["area_min_ha"])]
r["rend"] = r["producao_t"] * 1000 / r["area_colhida_ha"]
r = r[r["rend"].between(1, P["rendimento_max"])]
r = r[r.groupby("CD_MICRO")["ano"].transform("size") >= P["min_anos_micro"]]


def tendencia(g: pd.DataFrame, metodo: str) -> pd.Series:
    anos, y = g["ano"].to_numpy(float), g["rend"].to_numpy(float)
    if metodo in ("lowess", "lowess15"):
        janela, it = (P["janela_tendencia_anos"], 3) if metodo == "lowess" else (15, 1)
        frac = min(1.0, janela / (anos.max() - anos.min() + 1))
        return pd.Series(lowess(y, anos, frac=frac, it=it, return_sorted=False), index=g.index)
    if metodo == "mediana9":
        s = pd.Series(y, index=anos).reindex(np.arange(anos.min(), anos.max() + 1))
        return pd.Series(s.rolling(9, center=True, min_periods=5).median().reindex(anos).to_numpy(), index=g.index)
    if metodo == "poli2":
        c = np.polyfit(anos - anos.mean(), y, 2)
        return pd.Series(np.polyval(c, anos - anos.mean()), index=g.index)


for met in ("lowess", "lowess15", "mediana9", "poli2"):
    r[f"tend_{met}"] = r.groupby("CD_MICRO", group_keys=False)[["ano", "rend"]].apply(lambda g: tendencia(g, met))
    r[f"anom_{met}"] = 100 * (r["rend"] / r[f"tend_{met}"] - 1)


def por_bioma(col: str) -> pd.DataFrame:
    d = r.dropna(subset=[col, "bioma"])
    return d.groupby(["ano", "bioma"]).apply(lambda g: np.average(g[col], weights=g["area_colhida_ha"]),
                                             include_groups=False).unstack()


anom_b = por_bioma("anom_lowess")
n_micros = r.dropna(subset=["anom_lowess"]).groupby(["ano", "bioma"])["CD_MICRO"].nunique().unstack()
print(f"{r['CD_MICRO'].nunique()} microrregiões com pelo menos {P['min_anos_micro']} anos | microrregiões por bioma em 1980 / 2000 / 2020:")
n_micros.loc[[1980, 2000, 2020]]

In [ ]:
# Exemplo visual: rendimento e tendência (LOWESS) em microrregiões de cada bioma
exemplos = (r.groupby(["bioma", "CD_MICRO"])["producao_t"].sum().reset_index()
              .sort_values("producao_t").drop_duplicates("bioma", keep="last"))
fig, axes = plt.subplots(1, len(exemplos), figsize=(4.2 * len(exemplos), 3.4), layout="constrained")
nomes_micro = micro.set_index("CD_MICRO")["name_micro"]
for ax, (_, e) in zip(axes, exemplos.iterrows()):
    g = r[r["CD_MICRO"] == e["CD_MICRO"]].sort_values("ano")
    ax.plot(g["ano"], g["rend"], "o", ms=3, color="grey", label="rendimento")
    ax.plot(g["ano"], g["tend_lowess"], color="#1b7837", lw=2, label="LOWESS robusto, 25 anos (principal)")
    ax.plot(g["ano"], g["tend_lowess15"], color="#d95f02", lw=1.2, ls="--", label="LOWESS 15 anos (versão original)")
    ax.set_title(f"{e['bioma']}\n{nomes_micro.get(e['CD_MICRO'], e['CD_MICRO'])}", fontsize=9)
axes[0].set_ylabel("kg/ha"); axes[0].legend(fontsize=7)
plt.show()

fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
for b in anom_b.columns:
    ax.plot(anom_b.index, anom_b[b], marker=".", label=b)
ax.axhline(0, color="black", lw=0.7)
ax.set_ylabel("anomalia de rendimento (%)"); ax.set_title("Anomalia de rendimento por bioma (média ponderada pela área)")
ax.legend(fontsize=8, ncol=5)
plt.show()

### 2b. Sensibilidade ao método de tendência
Se a anomalia por bioma for parecida com as três tendências, as conclusões não dependem dessa escolha.

In [ ]:
sens_tend = {}
anom_alt = {met: por_bioma(f"anom_{met}") for met in ("lowess15", "mediana9", "poli2")}
for met, alt in anom_alt.items():
    sens_tend[f"corr. com {met}"] = {b: anom_b[b].corr(alt[b]) for b in anom_b.columns}
# efeito nas safras de La Niña de 2020–2023 (onde a tendência flexível absorvia o sinal)
for met, alt in {"lowess (principal)": anom_b, **anom_alt}.items():
    sens_tend[f"média 2021–2023 · {met}"] = {b: alt[b].reindex([2021, 2022, 2023]).mean() for b in anom_b.columns}
pd.DataFrame(sens_tend).round(2)

## 3. Perfil de janelas do ONI × rendimento
Para cada safra *t* (colheita em *t*), compara-se a anomalia de rendimento com o **ONI médio de 3 meses** terminando em cada mês, de **janeiro de *t*−1** (antes do plantio) até **abril de *t*** (colheita). São 16 janelas.

**Teste (fixado antes da execução):** em cada bioma, o **máximo |ρ| entre as 16 janelas** é comparado com o mesmo máximo calculado com **séries ONI substitutas** (como na seção 17 do notebook de defasagens): 2.000 randomizações de fase e os deslocamentos circulares por anos inteiros. **Passa** se p < 0,05 nas duas famílias, **indício** se em uma.

**Período:** safras de 1975 a 2025 (o ONI começa em 1950).

In [ ]:
pac = pd.read_parquet(PROCESSED / "pacifico_indices.parquet")
pac["mes_ref"] = pd.to_datetime(dict(year=pac["ano"], month=pac["mes"], day=1)).dt.to_period("M")
oni_full = pac.set_index("mes_ref")["oni"].sort_index()
oni_ref = oni_full[(oni_full.index.year >= 1950) & (oni_full.index.year <= 2025)]
assert not oni_ref.isna().any() and len(oni_ref) % 12 == 0

JANELAS = [(-1, m) for m in range(1, 13)] + [(0, m) for m in range(1, 5)]   # (ano relativo à colheita, mês final)
NOME_J = {(-1, m): f"{['jan','fev','mar','abr','mai','jun','jul','ago','set','out','nov','dez'][m-1]}(t−1)" for m in range(1, 13)}
NOME_J.update({(0, m): f"{['jan','fev','mar','abr'][m-1]}(t)" for m in range(1, 5)})
SAFRAS = np.arange(1975, P["ano_fim"] + 1)


def matriz_janelas(serie: pd.Series) -> np.ndarray:
    # safras × janelas: média de 3 meses terminando no mês indicado
    m3 = serie.rolling(3, min_periods=3).mean()
    out = np.empty((len(SAFRAS), len(JANELAS)))
    for i, s_ in enumerate(SAFRAS):
        for k, (da, mes) in enumerate(JANELAS):
            out[i, k] = m3.get(pd.Period(year=int(s_ + da), month=mes, freq="M"), np.nan)
    return out


def spearman_cols(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    ok = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    ry = rankdata(y[ok]); ry = (ry - ry.mean()) / ry.std()
    rx = rankdata(X[ok], axis=0); rx = (rx - rx.mean(0)) / rx.std(0)
    return (rx * ry[:, None]).mean(0)


def substitutas_fase(x: np.ndarray, n: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    media, F = x.mean(), np.fft.rfft(x - x.mean())
    amp, out = np.abs(F), np.empty((n, len(x)))
    for i in range(n):
        fase = rng.uniform(0, 2 * np.pi, len(F)); fase[0] = 0
        if len(x) % 2 == 0:
            fase[-1] = 0
        out[i] = np.fft.irfft(amp * np.exp(1j * fase), len(x)) + media
    return out


x_oni = oni_ref.to_numpy()
subs = {"fase": substitutas_fase(x_oni, P["n_fase"], P["seed"]),
        "deslocamento": np.array([np.roll(x_oni, 12 * k) for k in range(P["desloc_min_anos"], len(x_oni) // 12 - P["desloc_min_anos"] + 1)])}
X_obs = matriz_janelas(oni_ref)
X_subs = {fam: [matriz_janelas(pd.Series(s, index=oni_ref.index)) for s in arr] for fam, arr in subs.items()}

linhas, perfis = [], {}
for b in anom_b.columns:
    y = anom_b[b].reindex(SAFRAS).to_numpy()
    rho = spearman_cols(X_obs, y)
    perfis[b] = rho
    mx, k = np.nanmax(np.abs(rho)), int(np.nanargmax(np.abs(rho)))
    p_ = {fam: (1 + sum(np.nanmax(np.abs(spearman_cols(Xs, y))) >= mx for Xs in Xl)) / (len(Xl) + 1) for fam, Xl in X_subs.items()}
    n_sig = sum(v < 0.05 for v in p_.values())
    linhas.append({"bioma": b, "safras": int((~np.isnan(y)).sum()), "janela com máx|ρ|": NOME_J[JANELAS[k]], "ρ": rho[k],
                   "ρ ONI da safra (nov–jan)": rho[JANELAS.index((0, 1))],
                   "p fase": p_["fase"], "p deslocamento": p_["deslocamento"],
                   "resultado": "passa" if n_sig == 2 else ("indício" if n_sig == 1 else "não passa")})
tab_jan = pd.DataFrame(linhas).sort_values("p fase")
tab_jan.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.3), layout="constrained")
for b, rho in perfis.items():
    ax.plot(range(len(JANELAS)), rho, marker="o", label=b)
ax.axhline(0, color="black", lw=0.7)
ax.axvspan(JANELAS.index((-1, 9)) - 0.5, len(JANELAS) - 0.5, color="#f0f0f0", zorder=0)
ax.text(JANELAS.index((-1, 9)), ax.get_ylim()[1] * 0.9, "safra (set–abr)", fontsize=8)
ax.set_xticks(range(len(JANELAS)), [NOME_J[j_] for j_ in JANELAS], rotation=45)
ax.set_xlabel("mês final da janela de 3 meses do ONI"); ax.set_ylabel("ρ (ONI × anomalia de rendimento)")
ax.set_title(f"Perfil de janelas do ONI × rendimento por bioma (safras {SAFRAS[0]}–{SAFRAS[-1]})")
ax.legend(fontsize=8, ncol=5)
plt.show()

### 3b. Sensibilidade: o mesmo teste por UF (principais produtoras)
Prevista no D-01 (UF como divisão de sensibilidade). Média das anomalias das microrregiões da UF, ponderada pela área.

In [ ]:
uf_micro = micro.set_index("CD_MICRO")["abbrev_state"]
r["uf"] = r["CD_MICRO"].map(uf_micro)
UFS = ["MT", "PR", "RS", "GO", "MS", "BA", "MG", "SP", "SC", "TO", "MA", "PI"]
anom_uf = (r.dropna(subset=["anom_lowess"]).groupby(["ano", "uf"])
             .apply(lambda g: np.average(g["anom_lowess"], weights=g["area_colhida_ha"]), include_groups=False).unstack())


def teste_janelas(y_serie: pd.Series) -> dict:
    y = y_serie.reindex(SAFRAS).to_numpy()
    rho = spearman_cols(X_obs, y)
    mx, k = np.nanmax(np.abs(rho)), int(np.nanargmax(np.abs(rho)))
    p_ = {fam: (1 + sum(np.nanmax(np.abs(spearman_cols(Xs, y))) >= mx for Xs in Xl)) / (len(Xl) + 1) for fam, Xl in X_subs.items()}
    n_sig = sum(v < 0.05 for v in p_.values())
    return {"safras": int((~np.isnan(y)).sum()), "janela com máx|ρ|": NOME_J[JANELAS[k]], "ρ": rho[k],
            "p fase": p_["fase"], "p deslocamento": p_["deslocamento"],
            "resultado": "passa" if n_sig == 2 else ("indício" if n_sig == 1 else "não passa")}


tab_uf = pd.DataFrame([{"UF": u, **teste_janelas(anom_uf[u])} for u in UFS if u in anom_uf.columns])
tab_uf.sort_values("p fase").round(3)

## 4. Rendimento por fase do ENSO
A mesma classificação do D-06 (definição da NOAA; a safra é El Niño ou La Niña se pelo menos 4 dos 8 meses de set–abr estiverem na fase), agora para as safras de 1975 a 2025. A anomalia média de rendimento por fase tem IC de 95% por bootstrap e teste de permutação. **Robusto:** o IC não cruza o zero **e** p < 0,05.

In [ ]:
def fase_mensal(oni: pd.Series, limiar: float = 0.5, n_min: int = 5) -> pd.Series:
    out = pd.Series("neutro", index=oni.index)
    for nome, cond in (("El Niño", oni >= limiar), ("La Niña", oni <= -limiar)):
        grupo = (cond != cond.shift()).cumsum()
        out[cond & (cond.groupby(grupo).transform("size") >= n_min)] = nome
    return out


oni_m = oni_full.dropna()
fm = fase_mensal(oni_m)
safra_m = pd.Series(np.where(oni_m.index.month >= 9, oni_m.index.year + 1, oni_m.index.year), index=oni_m.index)
na_jan = np.isin(oni_m.index.month, [9, 10, 11, 12, 1, 2, 3, 4])
lin = []
for s_, idx in fm[na_jan].groupby(safra_m[na_jan]).groups.items():
    if s_ < SAFRAS[0] or s_ > SAFRAS[-1]:
        continue
    cont = fm[idx].value_counts()
    pico = oni_m[idx][oni_m[idx].abs().idxmax()]
    fase = "El Niño" if cont.get("El Niño", 0) >= 4 else ("La Niña" if cont.get("La Niña", 0) >= 4 else "neutro")
    inten = "" if fase == "neutro" else ("forte" if abs(pico) >= 1.5 else "moderado" if abs(pico) >= 1.0 else "fraco")
    lin.append(dict(safra=int(s_), rotulo=f"{s_ - 1}/{str(s_)[2:]}", fase=fase, intensidade=inten, oni_pico=pico))
fases = pd.DataFrame(lin).set_index("safra")
print(fases["fase"].value_counts().to_dict())


def comparar_fases(serie: pd.Series, n_boot: int = 5000, n_perm: int = P["n_perm"], seed: int = 42) -> dict:
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({"y": serie, "fase": fases["fase"]}).dropna()
    out = {}
    for f in ("El Niño", "La Niña", "neutro"):
        y = d.loc[d["fase"] == f, "y"].to_numpy()
        boot = [rng.choice(y, len(y)).mean() for _ in range(n_boot)]
        perm = np.array([rng.permutation(d["y"].to_numpy())[: len(y)].mean() for _ in range(n_perm)])
        p = (1 + (np.abs(perm - d["y"].mean()) >= abs(y.mean() - d["y"].mean())).sum()) / (n_perm + 1)
        out[f] = dict(media=y.mean(), ic_lo=np.percentile(boot, 2.5), ic_hi=np.percentile(boot, 97.5), p=p, n=len(y))
    en, ln = d.loc[d["fase"] == "El Niño", "y"].to_numpy(), d.loc[d["fase"] == "La Niña", "y"].to_numpy()
    obs, todos = en.mean() - ln.mean(), np.concatenate([en, ln])
    perm = np.array([(lambda z: z[: len(en)].mean() - z[len(en):].mean())(rng.permutation(todos)) for _ in range(n_perm)])
    out["dif"] = dict(media=obs, p=(1 + (np.abs(perm) >= abs(obs)).sum()) / (n_perm + 1))
    return out


linhas = []
for b in anom_b.columns:
    c = comparar_fases(anom_b[b].reindex(fases.index))
    l_ = {"bioma": b}
    for f in ("El Niño", "La Niña", "neutro"):
        rob = (c[f]["ic_lo"] > 0 or c[f]["ic_hi"] < 0) and c[f]["p"] < 0.05
        l_[f"{f} (%)"] = f"{c[f]['media']:+.1f} [{c[f]['ic_lo']:+.1f}; {c[f]['ic_hi']:+.1f}]" + (" *" if rob else "")
        l_[f"p {f}"] = c[f]["p"]
    l_["El Niño − La Niña (p.p.)"] = c["dif"]["media"]
    l_["p diferença"] = c["dif"]["p"]
    linhas.append(l_)
for u in ["RS", "PR", "MT", "GO"]:
    c = comparar_fases(anom_uf[u].reindex(fases.index))
    l_ = {"bioma": f"UF {u}"}
    for f in ("El Niño", "La Niña", "neutro"):
        rob = (c[f]["ic_lo"] > 0 or c[f]["ic_hi"] < 0) and c[f]["p"] < 0.05
        l_[f"{f} (%)"] = f"{c[f]['media']:+.1f} [{c[f]['ic_lo']:+.1f}; {c[f]['ic_hi']:+.1f}]" + (" *" if rob else "")
        l_[f"p {f}"] = c[f]["p"]
    l_["El Niño − La Niña (p.p.)"] = c["dif"]["media"]
    l_["p diferença"] = c["dif"]["p"]
    linhas.append(l_)
comp_rend = pd.DataFrame(linhas)
print("* = robusto (IC 95% não cruza zero e p < 0,05)")
comp_rend.round(3)

In [ ]:
cores = {"El Niño": "#c0392b", "neutro": "#999999", "La Niña": "#2471a3"}
fig, axes = plt.subplots(1, len(anom_b.columns), figsize=(4 * len(anom_b.columns), 3.6), sharey=False, layout="constrained")
for ax, b in zip(axes, anom_b.columns):
    d = pd.DataFrame({"oni": fases["oni_pico"], "y": anom_b[b].reindex(fases.index), "fase": fases["fase"], "rot": fases["rotulo"]}).dropna()
    for f, cor in cores.items():
        s_ = d[d["fase"] == f]
        ax.scatter(s_["oni"], s_["y"], color=cor, s=22, label=f)
    for _, l_ in d[(d["y"].abs() >= d["y"].abs().quantile(0.85))].iterrows():
        ax.annotate(l_["rot"], (l_["oni"], l_["y"]), fontsize=7, xytext=(3, 2), textcoords="offset points")
    ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
    ax.set_title(f"{b} · ρ = {stats.spearmanr(d['oni'], d['y']).statistic:+.2f} (n = {len(d)})", fontsize=10)
    ax.set_xlabel("ONI de pico na safra")
axes[0].set_ylabel("anomalia de rendimento (%)"); axes[0].legend(fontsize=7)
plt.show()

## 5. Estabilidade entre períodos
A relação é a mesma em 1975–1999 e em 2000–2025? A soja mudou muito de lugar nesse tempo (do Sul para o Cerrado) e o manejo também mudou. **Critério:** mesmo sinal nos dois períodos, na janela de máx|ρ| de cada bioma e na diferença El Niño − La Niña.

In [ ]:
linhas = []
for b in anom_b.columns:
    k = JANELAS.index(next(j_ for j_ in JANELAS if NOME_J[j_] == tab_jan.set_index("bioma").loc[b, "janela com máx|ρ|"]))
    l_ = {"bioma": b, "janela": NOME_J[JANELAS[k]]}
    for nome, (a0, a1) in {"1975–1999": (1975, 1999), "2000–2025": (2000, 2025)}.items():
        msk = (SAFRAS >= a0) & (SAFRAS <= a1)
        y = anom_b[b].reindex(SAFRAS).to_numpy()
        ok = msk & ~np.isnan(y)
        l_[f"ρ {nome}"] = stats.spearmanr(X_obs[ok, k], y[ok]).statistic if ok.sum() >= 8 else np.nan
        l_[f"n {nome}"] = int(ok.sum())
        sub = anom_b[b].reindex(fases.index)
        sub = sub[(sub.index >= a0) & (sub.index <= a1)]
        f_ = fases["fase"].reindex(sub.index)
        l_[f"EN − LN {nome} (p.p.)"] = sub[f_ == "El Niño"].mean() - sub[f_ == "La Niña"].mean()
    l_["mesmo sinal (ρ)"] = np.sign(l_["ρ 1975–1999"]) == np.sign(l_["ρ 2000–2025"])
    l_["mesmo sinal (EN − LN)"] = np.sign(l_["EN − LN 1975–1999 (p.p.)"]) == np.sign(l_["EN − LN 2000–2025 (p.p.)"])
    linhas.append(l_)
estab = pd.DataFrame(linhas)
estab.round(3)

### 5b. Período recente (2000–2025): **análise exploratória**
**Acrescentada depois de ver a tabela de estabilidade** (por isso, exploratória): no Cerrado a relação parece existir só no período recente. Este período coincide com a análise da chuva (2005 em diante), com a expansão da soja no Cerrado e com dados municipais menos afetados pelos desmembramentos. Os p-valores abaixo **não** são confirmatórios: o período foi escolhido depois de ver os dados.

In [ ]:
SAFRAS_REC = SAFRAS[SAFRAS >= 2000]
idx_rec = np.isin(SAFRAS, SAFRAS_REC)


def teste_recente(y_serie: pd.Series) -> dict:
    y = y_serie.reindex(SAFRAS).to_numpy().copy()
    y[~idx_rec] = np.nan
    rho = spearman_cols(X_obs, y)
    mx, k = np.nanmax(np.abs(rho)), int(np.nanargmax(np.abs(rho)))
    p_ = {fam: (1 + sum(np.nanmax(np.abs(spearman_cols(Xs, y))) >= mx for Xs in Xl)) / (len(Xl) + 1) for fam, Xl in X_subs.items()}
    return {"safras": int((~np.isnan(y)).sum()), "janela com máx|ρ|": NOME_J[JANELAS[k]], "ρ": rho[k],
            "p fase (exploratório)": p_["fase"], "p deslocamento (exploratório)": p_["deslocamento"]}


pd.DataFrame([{"região": b, **teste_recente(anom_b[b])} for b in anom_b.columns] +
             [{"região": f"UF {u}", **teste_recente(anom_uf[u])} for u in ["MT", "GO", "PR", "RS"]]).round(3)

## 6. Conclusões
**Dados:**
- **PAM por município, 1974–2025** (51 safras, de 1975 em diante), agregada por microrregião com o método do pipeline. A agregação **coincide em 99,98%** com a série oficial do IBGE por microrregião (1990–2025).
- **Limitação:** antes de 1990, os desmembramentos de municípios de fronteira não podem ser validados. Por isso a análise principal é por bioma, e a Caatinga, com poucas microrregiões e safras e uma tendência instável, **não deve ser interpretada**.

**Tendência:** LOWESS robusto com janela de 25 anos. A versão com 15 anos **absorvia o efeito das secas de vários anos**: no Pampa, a anomalia das safras de 2021–2023 (La Niña) era −4% com 15 anos e é −15% com 25 anos. Fora a Caatinga (instável com qualquer método), a correlação entre os métodos é ≥ 0,91.

**ENSO × rendimento na série inteira (1975–2025):**
| Região | Resultado | Leitura |
|---|---|---|
| **MATOPIBA (PI, MA)** | **passa** (ρ = −0,59 e −0,52) | El Niño → menor rendimento, coerente com o El Niño seco no Norte e no Nordeste |
| **MT** | indício (ρ = −0,36); fases: El Niño −3,0%, **La Niña +4,2% (robusto)**, diferença −7 p.p. (p = 0,01) | El Niño prejudica, La Niña favorece |
| **Cerrado (bioma)** | não passa (ρ = −0,31; p ≈ 0,13–0,15) | mesmo sentido do MT |
| **Amazônia** | quase (ρ = −0,32; p = 0,06 / 0,10) | El Niño → menor rendimento |
| **Pampa / RS / PR** | não passa (ρ ≈ +0,2) | sentido esperado (El Niño favorece), mas fraco |
| Mata Atlântica | sem relação | — |

**O que muda com o tempo (seção 5):**
- no **Cerrado**, a relação praticamente **não existia em 1975–1999** (ρ ≈ 0) e é **forte em 2000–2025** (ρ = −0,58);
- na análise exploratória do período recente, o Cerrado (ρ = −0,59) e o MT (ρ = −0,56) teriam p < 0,05 nas duas famílias de substitutas. **Esse resultado é exploratório:** o período foi escolhido depois de ver os dados;
- a mudança é coerente com a **expansão da soja no Cerrado** (de 4% da produção nacional em 1975 para 55% hoje) e com a qualidade maior dos dados municipais recentes.

**Comparação com a chuva** (notebook de defasagens):
1. **Cerrado:** o rendimento responde ao ENSO **de dezembro a abril da safra**, com o mesmo sentido da chuva (El Niño → menos chuva e rendimento menor). A relação com a chuva defasada de 7 meses (D-03, D-04) aparece aqui como uma relação com o ONI do fim do ano anterior e do início do ano da colheita.
2. **Pampa:** a chuva responde claramente aos eventos (La Niña −18%, D-06), mas o **rendimento responde pouco** ao ENSO como um todo. Muitas quebras de safra no Sul aconteceram também em anos neutros (anomalia média das neutras: −8%).

   Hipótese: o rendimento no Sul depende de **quando** a seca acontece (a floração e o enchimento de grãos), e não só do total da safra.
3. **Amazônia e MATOPIBA:** El Niño mais seco (D-06) e rendimento menor, uma relação coerente nas duas variáveis.

**Cautelas:**
- as correlações são moderadas (|ρ| entre 0,3 e 0,6): o ENSO explica uma parte da variação do rendimento, que também depende de pragas, preço, manejo e tecnologia;
- o resultado das UFs (PI e MA) vem de 12 UFs testadas: com a correção FDR, o p fica em torno de 0,04;
- a tendência usa a série inteira (análise descritiva). Para modelagem, ela deve ser ajustada só com os anos de treino.

**Para as próximas etapas:**
- o **período recente** (2000+) é o mais representativo da soja atual no Cerrado, e o sinal do ENSO nele é forte;
- o **MT e o MATOPIBA** têm a relação ENSO × rendimento mais clara;
- no **Sul**, o ENSO deve ser combinado com a chuva **por fase fenológica**, e não usado sozinho.